# 03 Run External Eval (S3PO-GS)

Goal:

1. use sparse-map ply as fixed scene representation;
2. run external evaluation on test split with two pose modes: gt and infer;
3. save rendered RGB/depth outputs and metrics json files;
4. inspect trajectory and ATE outputs for infer mode.

This notebook is an execution entry for:
- third_party/S3PO-GS/eval_external.py
- third_party/S3PO-GS/utils/external_eval_utils.py
- third_party/S3PO-GS/utils/external_pose_utils.py

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
import os
import json
import subprocess
from datetime import datetime
from pathlib import Path

# Force non-interactive matplotlib backend for headless cloud runs.
os.environ['MPLBACKEND'] = 'Agg'

import yaml

## 1. Paths and runtime selection

In [3]:
CV_ROOT = Path('/home/bzhang512/CV_Project')
S3PO_ROOT = CV_ROOT / 'third_party' / 'S3PO-GS'
NOTEBOOK_ROOT = CV_ROOT / 'part2_s3po' / 'notebooks'
CONFIG_ROOT = CV_ROOT / 'part2_s3po' / 'configs'
OUTPUT_ROOT = CV_ROOT / 'output' / 'part2_s3po'

PYTHON_CANDIDATES = [
    Path('/home/bzhang512/miniconda3/envs/s3po-gs/bin/python'),
    Path('/home/bzhang512/miniconda3/envs/S3PO-GS/bin/python'),
    Path('/home/bzhang512/miniconda3/bin/python'),
]

S3PO_PYTHON = None
for cand in PYTHON_CANDIDATES:
    if cand.exists():
        S3PO_PYTHON = cand
        break
if S3PO_PYTHON is None:
    S3PO_PYTHON = Path('python3')

print('S3PO_ROOT =', S3PO_ROOT)
print('CONFIG_ROOT =', CONFIG_ROOT)
print('OUTPUT_ROOT =', OUTPUT_ROOT)
print('S3PO_PYTHON =', S3PO_PYTHON)

S3PO_ROOT = /home/bzhang512/CV_Project/third_party/S3PO-GS
CONFIG_ROOT = /home/bzhang512/CV_Project/part2_s3po/configs
OUTPUT_ROOT = /home/bzhang512/CV_Project/output/part2_s3po
S3PO_PYTHON = /home/bzhang512/miniconda3/envs/s3po-gs/bin/python


## 2. Main experiment config

说明:
- `TEST_CONFIG_PATH` 使用 test split 配置；
- `PLY_PATH` 指向 sparse 建图结果（point_cloud.ply）；
- notebook 支持先跑 gt 再跑 infer，也可单独运行其中一个。

In [4]:
# -------- Required inputs --------
# Example choices:
#   CONFIG_ROOT / 's3po_re10k1_test.yaml'
#   CONFIG_ROOT / 's3po_dl3dv2_test.yaml'
TEST_CONFIG_PATH = CONFIG_ROOT / 's3po_re10k1_test.yaml'
# TEST_CONFIG_PATH = CONFIG_ROOT / 's3po_dl3dv2_test.yaml'
TEST_CONFIG_PATH = CONFIG_ROOT / 's3po_405841_test.yaml'

# Must be sparse stage map ply (fixed scene)
PLY_PATH = Path('/home/bzhang512/CV_Project/output/part2_s3po/re10k-1/s3po_re10k-1_sparse_full/Re10k-1_part2_s3po/2026-03-31-10-50-54/point_cloud/final/point_cloud.ply')
PLY_PATH = Path('/home/bzhang512/CV_Project/output/part2_s3po/dl3dv-2/s3po_dl3dv-2_sparse_full/DL3DV-2_part2_s3po/2026-04-02-21-25-38/point_cloud/final/point_cloud.ply')
#PLY_PATH = Path('/home/bzhang512/CV_Project/output/part2_s3po/405841/s3po_405841_sparse_full/405841_part2_s3po/2026-04-02-19-51-20/point_cloud/final/point_cloud.ply')
PLY_PATH = Path('/home/bzhang512/my_storage_500G/CV_Project/output/part2_s3po_previous/405841/s3po_405841_full_full/405841_part2_s3po/2026-04-03-12-48-39/point_cloud/final/point_cloud.ply')
# -------- Optional controls --------
RUN_GT = False
RUN_INFER = True

ORIGIN_MODE = 'test_to_sparse_first'   # 'none' or 'test_to_sparse_first'
INFER_INIT_MODE = 'gt_first'           # 'gt_first' or 'identity'

BEGIN_OVERRIDE = None
END_OVERRIDE = None

SAVE_RENDER_RGB = True
SAVE_RENDER_DEPTH = True
SAVE_RENDER_DEPTH_NPY = True

# Extract dataset tag from config filename: token between first and second '_'
# Example: s3po_re10k1_test.yaml -> re10k1
_config_stem = TEST_CONFIG_PATH.stem
_config_parts = _config_stem.split('_')
if len(_config_parts) >= 3:
    EXTERNAL_TAG = _config_parts[1]
else:
    EXTERNAL_TAG = _config_stem

EXTERNAL_SAVE_BASE = OUTPUT_ROOT / 'external_eval'
GT_SAVE_DIR = EXTERNAL_SAVE_BASE / EXTERNAL_TAG / f'gt_{EXTERNAL_TAG}'
INFER_SAVE_DIR = EXTERNAL_SAVE_BASE / EXTERNAL_TAG / f'infer_{EXTERNAL_TAG}_2'

print('TEST_CONFIG_PATH =', TEST_CONFIG_PATH)
print('PLY_PATH =', PLY_PATH)
print('EXTERNAL_TAG =', EXTERNAL_TAG)
print('GT_SAVE_DIR =', GT_SAVE_DIR)
print('INFER_SAVE_DIR =', INFER_SAVE_DIR)

TEST_CONFIG_PATH = /home/bzhang512/CV_Project/part2_s3po/configs/s3po_405841_test.yaml
PLY_PATH = /home/bzhang512/my_storage_500G/CV_Project/output/part2_s3po_previous/405841/s3po_405841_full_full/405841_part2_s3po/2026-04-03-12-48-39/point_cloud/final/point_cloud.ply
EXTERNAL_TAG = 405841
GT_SAVE_DIR = /home/bzhang512/CV_Project/output/part2_s3po/external_eval/405841/gt_405841
INFER_SAVE_DIR = /home/bzhang512/CV_Project/output/part2_s3po/external_eval/405841/infer_405841_2


## 3. Validate inputs and preview test config

In [5]:
assert S3PO_ROOT.exists(), f'Missing S3PO root: {S3PO_ROOT}'
assert (S3PO_ROOT / 'eval_external.py').exists(), 'Missing eval_external.py entry'
assert TEST_CONFIG_PATH.exists(), f'Missing test config: {TEST_CONFIG_PATH}'
assert PLY_PATH.exists(), f'Missing sparse map ply: {PLY_PATH}'

cfg = yaml.safe_load(TEST_CONFIG_PATH.read_text(encoding='utf-8'))
dataset_path = cfg['Dataset']['dataset_path']
dataset_type = cfg['Dataset']['type']
dataset_sensor = cfg['Dataset']['sensor_type']
cfg_begin = cfg['Dataset'].get('begin', None)
cfg_end = cfg['Dataset'].get('end', None)

print('dataset_path =', dataset_path)
print('dataset_type =', dataset_type)
print('dataset_sensor =', dataset_sensor)
print('config begin/end =', cfg_begin, cfg_end)
if BEGIN_OVERRIDE is not None or END_OVERRIDE is not None:
    print('override begin/end =', BEGIN_OVERRIDE, END_OVERRIDE)

dataset_path = /home/bzhang512/CV_Project/dataset/405841/part2_s3po/test
dataset_type = waymo
dataset_sensor = monocular
config begin/end = 0 179


## 4. Build command helpers

逻辑说明:
- gt: 使用对齐后的 GT 位姿直接渲染评测；
- infer: 使用 MASt3R+PnP 顺序推理位姿，再计算 ATE 并渲染评测；
- 两者都会产出 eval_external.json，且可保存渲染图/深度。

In [6]:
def build_eval_cmd(pose_mode: str, save_dir: Path):
    cmd = [
        str(S3PO_PYTHON),
        '/home/bzhang512/CV_Project/third_party/S3PO-GS/eval_external.py',
        '--config', str(TEST_CONFIG_PATH),
        '--ply_path', str(PLY_PATH),
        '--save_dir', str(save_dir),
        '--pose_mode', pose_mode,
        '--origin_mode', ORIGIN_MODE,
        '--infer_init_mode', INFER_INIT_MODE,
    ]

    if BEGIN_OVERRIDE is not None:
        cmd += ['--begin', str(BEGIN_OVERRIDE)]
    if END_OVERRIDE is not None:
        cmd += ['--end', str(END_OVERRIDE)]

    if not SAVE_RENDER_RGB:
        cmd += ['--no_save_render_rgb']
    if SAVE_RENDER_DEPTH:
        cmd += ['--save_render_depth']
    if SAVE_RENDER_DEPTH_NPY:
        cmd += ['--save_render_depth_npy']

    return cmd


def run_eval_cmd(cmd, title='external eval'):
    print(f'[{title}] command =')
    print(' '.join(cmd))

    env = os.environ.copy()
    env['MPLBACKEND'] = 'Agg'

    proc = subprocess.run(
        cmd,
        cwd=str(S3PO_ROOT),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(proc.stdout)
    print(f'[{title}] returncode =', proc.returncode)
    if proc.returncode != 0:
        raise RuntimeError(f'{title} failed with code={proc.returncode}')
    return proc

## 5. Run GT mode

GT mode output highlights:
- eval_external.json (metrics + per-frame records)
- eval_external_meta.json
- render_rgb / render_depth / render_depth_npy (按开关)

In [7]:
if RUN_GT:
    GT_SAVE_DIR.mkdir(parents=True, exist_ok=True)
    gt_cmd = build_eval_cmd('gt', GT_SAVE_DIR)
    _ = run_eval_cmd(gt_cmd, title='GT mode')
else:
    print('RUN_GT=False: skip GT mode')

RUN_GT=False: skip GT mode


In [8]:
gt_eval_json = GT_SAVE_DIR / 'eval_external.json'
gt_meta_json = GT_SAVE_DIR / 'eval_external_meta.json'

if gt_eval_json.exists():
    gt_eval = json.loads(gt_eval_json.read_text(encoding='utf-8'))
    print('GT avg_psnr =', gt_eval.get('avg_psnr', None))
    print('GT avg_ssim =', gt_eval.get('avg_ssim', None))
    print('GT avg_lpips =', gt_eval.get('avg_lpips', None))
    print('GT num_frames =', gt_eval.get('num_frames', None))
else:
    print('GT eval json not found:', gt_eval_json)

print('GT meta exists =', gt_meta_json.exists())
print('GT render_rgb exists =', (GT_SAVE_DIR / 'render_rgb').exists())
print('GT render_depth exists =', (GT_SAVE_DIR / 'render_depth').exists())
print('GT render_depth_npy exists =', (GT_SAVE_DIR / 'render_depth_npy').exists())

GT eval json not found: /home/bzhang512/CV_Project/output/part2_s3po/external_eval/405841/gt_405841/eval_external.json
GT meta exists = False
GT render_rgb exists = False
GT render_depth exists = False
GT render_depth_npy exists = False


## 6. Run Infer mode

Infer mode extra outputs:
- pose summary in eval_external.json -> pose
- plot/trj_external_infer.json
- plot/stats_external_infer.json
- plot/evo_2dplot_external_infer.png

In [9]:
if RUN_INFER:
    INFER_SAVE_DIR.mkdir(parents=True, exist_ok=True)
    infer_cmd = build_eval_cmd('infer', INFER_SAVE_DIR)
    _ = run_eval_cmd(infer_cmd, title='INFER mode')
else:
    print('RUN_INFER=False: skip Infer mode')

[INFER mode] command =
/home/bzhang512/miniconda3/envs/s3po-gs/bin/python /home/bzhang512/CV_Project/third_party/S3PO-GS/eval_external.py --config /home/bzhang512/CV_Project/part2_s3po/configs/s3po_405841_test.yaml --ply_path /home/bzhang512/my_storage_500G/CV_Project/output/part2_s3po_previous/405841/s3po_405841_full_full/405841_part2_s3po/2026-04-03-12-48-39/point_cloud/final/point_cloud.ply --save_dir /home/bzhang512/CV_Project/output/part2_s3po/external_eval/405841/infer_405841_2 --pose_mode infer --origin_mode test_to_sparse_first --infer_init_mode gt_first --save_render_depth --save_render_depth_npy
Warning, cannot find cuda-compiled version of RoPE2D, using a slow pytorch version instead
Current backend: agg
Eval: Could not compute test-to-sparse origin offset; fallback to no offset.
... loading model from naver/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric
instantiating : AsymmetricMASt3R(enc_depth=24, dec_depth=12, enc_embed_dim=1024, dec_embed_dim=768, enc_num_heads=16, de

In [10]:
infer_eval_json = INFER_SAVE_DIR / 'eval_external.json'
infer_meta_json = INFER_SAVE_DIR / 'eval_external_meta.json'
infer_plot_dir = INFER_SAVE_DIR / 'plot'
infer_trj_json = infer_plot_dir / 'trj_external_infer.json'
infer_stats_json = infer_plot_dir / 'stats_external_infer.json'
infer_evo_png = infer_plot_dir / 'evo_2dplot_external_infer.png'

if infer_eval_json.exists():
    infer_eval = json.loads(infer_eval_json.read_text(encoding='utf-8'))
    print('Infer avg_psnr =', infer_eval.get('avg_psnr', None))
    print('Infer avg_ssim =', infer_eval.get('avg_ssim', None))
    print('Infer avg_lpips =', infer_eval.get('avg_lpips', None))
    print('Infer num_frames =', infer_eval.get('num_frames', None))

    pose_info = infer_eval.get('pose', {})
    print('Infer ate_rmse =', pose_info.get('ate_rmse', None))
    print('Infer pose_success_rate =', pose_info.get('pose_success_rate', None))
    print('Infer pose_success_count =', pose_info.get('pose_success_count', None))
    print('Infer pose_total_count =', pose_info.get('pose_total_count', None))
else:
    print('Infer eval json not found:', infer_eval_json)

print('Infer meta exists =', infer_meta_json.exists())
print('Infer trajectory exists =', infer_trj_json.exists())
print('Infer evo stats exists =', infer_stats_json.exists())
print('Infer evo plot exists =', infer_evo_png.exists())
print('Infer render_rgb exists =', (INFER_SAVE_DIR / 'render_rgb').exists())
print('Infer render_depth exists =', (INFER_SAVE_DIR / 'render_depth').exists())
print('Infer render_depth_npy exists =', (INFER_SAVE_DIR / 'render_depth_npy').exists())

Infer avg_psnr = 17.600319936954776
Infer avg_ssim = 0.5847837014238262
Infer avg_lpips = 0.4289688354763905
Infer num_frames = 179
Infer ate_rmse = 1.0749489976007145
Infer pose_success_rate = 1.0
Infer pose_success_count = 179
Infer pose_total_count = 179
Infer meta exists = True
Infer trajectory exists = True
Infer evo stats exists = True
Infer evo plot exists = True
Infer render_rgb exists = True
Infer render_depth exists = True
Infer render_depth_npy exists = True


## 7. Quick comparison summary

In [11]:
def _safe_read_json(path: Path):
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding='utf-8'))

gt_eval = _safe_read_json(GT_SAVE_DIR / 'eval_external.json')
infer_eval = _safe_read_json(INFER_SAVE_DIR / 'eval_external.json')

if gt_eval is None:
    print('GT result is missing')
else:
    print('GT  : PSNR={:.4f} SSIM={:.4f} LPIPS={:.4f} N={}'.format(
        float(gt_eval.get('avg_psnr', 0.0)),
        float(gt_eval.get('avg_ssim', 0.0)),
        float(gt_eval.get('avg_lpips', 0.0)),
        int(gt_eval.get('num_frames', 0)),
    ))

if infer_eval is None:
    print('Infer result is missing')
else:
    pose_info = infer_eval.get('pose', {})
    print('Infer: PSNR={:.4f} SSIM={:.4f} LPIPS={:.4f} N={} ATE={}'.format(
        float(infer_eval.get('avg_psnr', 0.0)),
        float(infer_eval.get('avg_ssim', 0.0)),
        float(infer_eval.get('avg_lpips', 0.0)),
        int(infer_eval.get('num_frames', 0)),
        pose_info.get('ate_rmse', None),
    ))

print('GT output dir    =', GT_SAVE_DIR)
print('Infer output dir =', INFER_SAVE_DIR)

GT result is missing
Infer: PSNR=17.6003 SSIM=0.5848 LPIPS=0.4290 N=179 ATE=1.0749489976007145
GT output dir    = /home/bzhang512/CV_Project/output/part2_s3po/external_eval/405841/gt_405841
Infer output dir = /home/bzhang512/CV_Project/output/part2_s3po/external_eval/405841/infer_405841_2
